In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder, TargetEncoder, MinMaxScaler
from xgboost import XGBClassifier
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt
import seaborn as sns

ModuleNotFoundError: No module named 'numpy'

In [2]:
dados = pd.read_parquet('../../data/dados_modelo/base_modelo.parquet')
dados.head()

,id_municipio,id_escola,chave_rede,alfabetizado,proficiencia,peso_aluno,ano,taxa_alfabetizacao_mun,media_portugues,sigla_uf,taxa_alfabetizacao_uf,indice_analf,MEDIA_INSE,razao_inse_analf,raz_alfab_mun_uf,media_inse_uf,diferenca_inse_uf
0,5101837,None,None,None,NaN,NaN,<NA>,NaN,NaN,MT,NaN,8.46,NaN,NaN,NaN,4.993129,NaN
1,1100205,60000002,2,Sim,746.91,1.00,2023,57.45,747.2139,RO,58.65,4.36,4.959,1.137383,0.97954,4.887929,0.071071
2,1100205,60000386,2,Não,727.29,1.00,2023,57.45,747.2139,RO,58.65,4.36,4.959,1.137383,0.97954,4.887929,0.071071
3,1100205,60000386,2,Sim,764.40,1.00,2023,57.45,747.2139,RO,58.65,4.36,4.959,1.137383,0.97954,4.887929,0.071071
4,1100205,60000002,2,Não,739.06,1.05,2023,57.45,747.2139,RO,58.65,4.36,4.959,1.137383,0.97954,4.887929,0.071071


In [3]:
dados.isna().sum()

id_municipio                 0
id_escola                 5140
chave_rede                  21
alfabetizado              5140
proficiencia              5140
peso_aluno                5140
ano                         21
taxa_alfabetizacao_mun      21
media_portugues             21
sigla_uf                     0
taxa_alfabetizacao_uf       21
indice_analf                 0
MEDIA_INSE                1467
razao_inse_analf          1467
raz_alfab_mun_uf            21
media_inse_uf               15
diferenca_inse_uf         1467
dtype: int64

In [4]:
df = dados.copy()

In [5]:
df = df[df['alfabetizado'].notna()]
df.isna().sum()


id_municipio                 0
id_escola                    0
chave_rede                   0
alfabetizado                 0
proficiencia                 0
peso_aluno                   0
ano                          0
taxa_alfabetizacao_mun       0
media_portugues              0
sigla_uf                     0
taxa_alfabetizacao_uf        0
indice_analf                 0
MEDIA_INSE                1429
razao_inse_analf          1429
raz_alfab_mun_uf             0
media_inse_uf                0
diferenca_inse_uf         1429
dtype: int64

In [6]:
dir_uf = pd.read_parquet('../../data/dir_uf.parquet')
dir_uf

,id_uf,sigla_uf
0,51,MT
1,11,RO
2,12,AC
3,13,AM
4,14,RR
5,15,PA
6,16,AP
7,17,TO
8,21,MA
9,22,PI


Apenas a coluna "MEDIA_INSE" contém dados vazios. Acertaremos isso no pipeline, no momento de imputação dos valores restantes.

In [7]:
# Mapeamento do Alvo (Target Encoding Binário)
if 'alfabetizado' in df.columns:
    target_map = {'Sim': 1, 'Não': 0}
    df['alfabetizado'] = df['alfabetizado'].map(target_map)

In [8]:
uf_map = dir_uf.set_index('sigla_uf')['id_uf'].to_dict()
    
# Submeter o mapeamento e criar (ou substituir) a coluna
df['id_uf'] = df['sigla_uf'].map(uf_map)
df['id_uf'] = pd.to_numeric(df['id_uf'], errors='coerce')

# (Opcional) Se quiser remover a sigla antiga e manter só o ID:
df = df.drop(columns=['sigla_uf'])

df

,id_municipio,id_escola,chave_rede,alfabetizado,proficiencia,peso_aluno,ano,taxa_alfabetizacao_mun,media_portugues,taxa_alfabetizacao_uf,indice_analf,MEDIA_INSE,razao_inse_analf,raz_alfab_mun_uf,media_inse_uf,diferenca_inse_uf,id_uf
1,1100205,60000002,2,1,746.91,1.00,2023,57.45,747.2139,58.65,4.36,4.959,1.137383,0.979540,4.887929,0.071071,11
2,1100205,60000386,2,0,727.29,1.00,2023,57.45,747.2139,58.65,4.36,4.959,1.137383,0.979540,4.887929,0.071071,11
3,1100205,60000386,2,1,764.40,1.00,2023,57.45,747.2139,58.65,4.36,4.959,1.137383,0.979540,4.887929,0.071071,11
4,1100205,60000002,2,0,739.06,1.05,2023,57.45,747.2139,58.65,4.36,4.959,1.137383,0.979540,4.887929,0.071071,11
5,1100205,60000012,2,0,736.43,1.21,2023,57.45,747.2139,58.65,4.36,4.959,1.137383,0.979540,4.887929,0.071071,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1821405,5220405,60041783,3,1,758.16,1.11,2023,72.78,757.9946,66.72,8.28,5.413,0.653743,1.090827,5.067078,0.345922,52
1821406,5220405,60041698,3,1,756.65,1.14,2023,72.78,757.9946,66.72,8.28,5.413,0.653743,1.090827,5.067078,0.345922,52
1821407,5220405,60041698,3,1,772.79,1.15,2023,72.78,757.9946,66.72,8.28,5.413,0.653743,1.090827,5.067078,0.345922,52
1821408,5220405,60041698,3,1,771.91,1.00,2023,72.78,757.9946,66.72,8.28,5.413,0.653743,1.090827,5.067078,0.345922,52


In [9]:
# Remoção de Colunas Indesejadas e Vazamentos de Dados (Data Leakage)
cols_para_remover = [
        'id_escola',        # Vimos que este dado muda de um ano para outro
        'proficiencia',     # VAZAMENTO ABSOLUTO: Nota do próprio aluno no exame
        'peso_aluno',       # Peso estatístico amostral, sem valor preditivo individual
        'media_portugues',  # Nota já compõe o índice de proficiência, traz vazamento.
        'ano',
        'id_municipio'
    ]

In [10]:
# Garante remoção apenas das colunas existentes na base
existing_cols_to_drop = [col for col in cols_para_remover if col in df.columns]
df = df.drop(columns=existing_cols_to_drop)

In [11]:
# Imputa os valores ausentes com a mediana por UF
df['MEDIA_INSE'] = df['MEDIA_INSE'].fillna(
    df.groupby('id_uf')['MEDIA_INSE'].transform('median')
)

# Imputa os valores ausentes com a mediana por UF
df['diferenca_inse_uf'] = df['diferenca_inse_uf'].fillna(
    df.groupby('id_uf')['diferenca_inse_uf'].transform('median')
)

df['razao_inse_analf'] = df['razao_inse_analf'].fillna(
    df.groupby('id_uf')['razao_inse_analf'].transform('median')
)

In [12]:
df.isna().sum()

chave_rede                0
alfabetizado              0
taxa_alfabetizacao_mun    0
taxa_alfabetizacao_uf     0
indice_analf              0
MEDIA_INSE                0
razao_inse_analf          0
raz_alfab_mun_uf          0
media_inse_uf             0
diferenca_inse_uf         0
id_uf                     0
dtype: int64

Resolvido o problema da coluna MEDIA_INSE

In [13]:
# Seleção das colunas após limpeza
colunas_numericas = ['taxa_alfabetizacao_mun','taxa_alfabetizacao_uf', 'indice_analf', 'MEDIA_INSE', 'razao_inse_analf',
                     'raz_alfab_mun_uf', 'media_inse_uf', 'diferenca_inse_uf']
cat_baixa_card = ['chave_rede']
cat_alta_card = ['id_uf']
coluna_alvo = 'alfabetizado'

In [14]:
from scipy.stats import spearmanr, pointbiserialr, chi2_contingency
from sklearn.feature_selection import mutual_info_classif

In [15]:
# Pipeline Numérico: Imputação por Mediana + Escalonamento Robusto contra Outliers
num_transformer = Pipeline(steps=[
        ('scaler', MinMaxScaler())
    ])

# Pipeline Categórico para Baixa Cardinalidade: Imputação por Moda + One-Hot Encoding
cat_low_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist', sparse_output=False))
])

# Pipeline Categórico para Alta Cardinalidade: Imputação por Moda + Target Encoding
cat_high_transformer = Pipeline(steps=[
    ('target_enc', TargetEncoder(smooth="auto", cv=5))
])

In [16]:
# Montagem do ColumnTransformer Unificado
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, colunas_numericas),
        ('cat_low', cat_low_transformer, cat_baixa_card),
        ('cat_high', cat_high_transformer, cat_alta_card)
    ],
    remainder='drop'
)

In [17]:
X = df.drop(columns=[coluna_alvo])
y = df[coluna_alvo]

In [18]:
# Divisão Treino/Teste Estratificada (ANTES de qualquer imputação ou transformação)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

In [19]:
# --- A. Teste de Correlação (Spearman e Pearson/Point-Biserial) ---
correlations = []
for col in colunas_numericas:
    spearman_corr, spearman_p = spearmanr(X_train[col], y_train)
    pb_corr, pb_p = pointbiserialr(X_train[col], y_train)
    
    correlations.append({
        'Feature': col,
        'Pearson/PointBiserial': pb_corr,
        'p-valor (Pearson)': pb_p,
        'Spearman': spearman_corr,
        'p-valor (Spearman)': spearman_p
    })

df_corr = pd.DataFrame(correlations)
print("=== CORRELAÇÃO DE VARIÁVEIS NUMÉRICAS ===")
print(df_corr.sort_values(by='Spearman', key=abs, ascending=False))

# --- B. Teste Qui-Quadrado para Categóricas ---
chi2_results = []
for col in cat_baixa_card:
    contingency_table = pd.crosstab(X_train[col].astype(str), y_train)
    chi2, p, dof, _ = chi2_contingency(contingency_table)
    chi2_results.append({
        'Feature': col,
        'Chi2 Statistic': chi2,
        'p-valor': p
    })

df_chi2 = pd.DataFrame(chi2_results)
print("\n=== TESTE QUI-QUADRADO (CATEGÓRICAS) ===")
print(df_chi2)

# --- C. Mutual Information (Numéricas + Categóricas) ---
# Prepara dados para o Mutual Info (exige tratar NaNs)
X_train_mi = X_train.copy()
mi_scores = mutual_info_classif(X_train_mi, y_train, random_state=42)

df_mi = pd.DataFrame({
    'Feature': X_train_mi.columns,
    'Mutual Information': mi_scores
}).sort_values(by='Mutual Information', ascending=False)

print("\n=== MUTUAL INFORMATION ===")
print(df_mi)

=== CORRELAÇÃO DE VARIÁVEIS NUMÉRICAS ===
                  Feature  Pearson/PointBiserial  p-valor (Pearson)  Spearman  \
0  taxa_alfabetizacao_mun               0.245809           0.000000  0.239046   
1   taxa_alfabetizacao_uf               0.200612           0.000000  0.189183   
5        raz_alfab_mun_uf               0.142893           0.000000  0.148203   
3              MEDIA_INSE               0.063451           0.000000  0.062476   
7       diferenca_inse_uf               0.052317           0.000000  0.059106   
6           media_inse_uf               0.037110           0.000000  0.006940   
2            indice_analf              -0.002618           0.001604  0.003601   
4        razao_inse_analf               0.001751           0.034770  0.003401   

   p-valor (Spearman)  
0        0.000000e+00  
1        0.000000e+00  
5        0.000000e+00  
3        0.000000e+00  
7        0.000000e+00  
6        5.963699e-17  
2        1.422611e-05  
4        4.131759e-05  

=== TESTE Q

In [20]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

# 1. PASSO 1: Definir o seu full_pipeline (exatamente como você já fez)
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        random_state=42,
        eval_metric='logloss',  # Evita avisos de depreciação nas versões recentes
        n_jobs=-1
    ))
])

# Grade de distribuição de hiperparâmetros
param_distributions_xgb = {
    # Taxa de aprendizado (learning rate)
    'classifier__learning_rate': uniform(0.01, 0.2),       # Amplitude de 0.01 a 0.21
    
    # Profundidade e complexidade da árvore
    'classifier__max_depth': randint(3, 10),               # Profundidade máxima das árvores
    'classifier__min_child_weight': randint(1, 10),        # Equivalente ao peso mínimo para divisão de nós
    'classifier__gamma': uniform(0, 0.5),                  # Redução mínima de perda para fazer uma divisão
    
    # Número de estimadores (árvores)
    'classifier__n_estimators': randint(100, 300),         # Quantidade de árvores de decisão
    
    # Amostragem (regularização e ruído)
    'classifier__subsample': uniform(0.6, 0.4),            # Proporção de linhas amostradas por árvore (0.6 a 1.0)
    'classifier__colsample_bytree': uniform(0.6, 0.4),     # Proporção de colunas amostradas por árvore (0.6 a 1.0)
    
    'classifier__reg_alpha': uniform(0, 1.0),            # L1 (Lasso)
    'classifier__reg_lambda': uniform(1.0, 2.0),          # L2 (Ridge)
    
    'classifier__scale_pos_weight': [1.0, 1.40, 1.8, 2.0]
}

# 3. PASSO 3: Passar o seu full_pipeline para o RandomizedSearchCV
random_search_xgb = RandomizedSearchCV(
    estimator=full_pipeline,
    param_distributions=param_distributions_xgb,
    n_iter=30,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

# 4. PASSO 4: Treinar e buscar os melhores hiperparâmetros
random_search_xgb.fit(X_train, y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'classifier__colsample_bytree': <scipy.stats....00182606F3950>, 'classifier__gamma': <scipy.stats....0018260BBDF90>, 'classifier__learning_rate': <scipy.stats....0018260AD16A0>, 'classifier__max_depth': <scipy.stats....0018260AD1A90>, ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",30
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the sa

In [21]:
# 4. Exibir os melhores resultados encontrados na Validação Cruzada
print("\n=== MELHORES RESULTADOS ===")
print(f"Melhor ROC-AUC na CV: {random_search_xgb.best_score_:.4f}")
print("Melhores Hiperparâmetros:")
for param, value in random_search_xgb.best_params_.items():
    print(f"  - {param}: {value}")


=== MELHORES RESULTADOS ===
Melhor ROC-AUC na CV: 0.7534
Melhores Hiperparâmetros:
  - classifier__colsample_bytree: 0.749816047538945
  - classifier__gamma: 0.4753571532049581
  - classifier__learning_rate: 0.15639878836228102
  - classifier__max_depth: 7
  - classifier__min_child_weight: 5
  - classifier__n_estimators: 202
  - classifier__reg_alpha: 0.44583275285359114
  - classifier__reg_lambda: 1.1999498316360058
  - classifier__scale_pos_weight: 1.8
  - classifier__subsample: 0.9464704583099741


In [22]:
from sklearn.metrics import classification_report, roc_auc_score
from scipy.stats import randint, uniform

# 5. Avaliação do Melhor Modelo no Conjunto de Teste (Dados Inéditos)
# O 'search.best_estimator_' já re-treinou o modelo com a base de treino completa usando os melhores parâmetros
best_model = random_search_xgb.best_estimator_

y_pred_test = best_model.predict(X_test)
y_proba_test = best_model.predict_proba(X_test)[:, 1]

print("\n=== DESEMPENHO NO CONJUNTO DE TESTE ===")
print(f"ROC-AUC Teste: {roc_auc_score(y_test, y_proba_test):.4f}")
print("\nRelatório de Classificação:")
print(classification_report(y_test, y_pred_test, target_names=['Não Alfabetizado', 'Alfabetizado']))


=== DESEMPENHO NO CONJUNTO DE TESTE ===
ROC-AUC Teste: 0.6793

Relatório de Classificação:
                  precision    recall  f1-score   support

Não Alfabetizado       0.68      0.11      0.18    145928
    Alfabetizado       0.62      0.97      0.75    217326

        accuracy                           0.62    363254
       macro avg       0.65      0.54      0.47    363254
    weighted avg       0.64      0.62      0.52    363254



In [23]:
# Esse é o modelo final otimizado e treinado:
modelo_final = random_search_xgb.best_estimator_

# Salvá-lo em disco para usar depois sem precisar retreinar:
import joblib

joblib.dump(modelo_final, 'modelo_xgboost_otimizado.pkl')
print("✅ Modelo final salvo com sucesso!")

✅ Modelo final salvo com sucesso!


In [24]:
# 1. Pega o melhor modelo do Random Search
best_model = random_search_xgb.best_estimator_

# 2. Altera/Adiciona parâmetros diretamente nele
best_model.set_params(
    scale_pos_weight=1.48,  # Ajusta o peso das classes
    learning_rate=0.05      # Exemplo: reduz a taxa de aprendizado
)

# 3. RE-TREINA o modelo ajustado na base de treino completa
best_model.fit(X_train, y_train)

# 4. Avalia novamente no teste
y_pred_novo = best_model.predict(X_test)
y_proba_test = best_model.predict_proba(X_test)[:, 1]

print("\n=== DESEMPENHO NO CONJUNTO DE TESTE ===")
print(f"ROC-AUC Teste: {roc_auc_score(y_test, y_proba_test):.4f}")
print("\nRelatório de Classificação:")
print(classification_report(y_test, y_pred_test, target_names=['Não Alfabetizado', 'Alfabetizado']))

ValueError: Invalid parameter 'scale_pos_weight' for estimator Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   MinMaxScaler())]),
                                                  ['taxa_alfabetizacao_mun',
                                                   'taxa_alfabetizacao_uf',
                                                   'indice_analf', 'MEDIA_INSE',
                                                   'razao_inse_analf',
                                                   'raz_alfab_mun_uf',
                                                   'media_inse_uf',
                                                   'diferenca_inse_uf']),
                                                 ('cat_low',
                                                  Pipeline(steps=[('onehot',
                            